In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="mixtral-8x7b-32768", groq_api_key=groq_api_key)

c:\Users\91977\Desktop\gen ai\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [11]:
#tool create
from langchain_core.tools import tool
import requests

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    Fetches the conversion factor between a base currency and a target currency.

    Args:
        base_currency: Three-letter ISO currency code (e.g., USD)
        target_currency: Three-letter ISO currency code (e.g., INR)

    Returns:
        Conversion factor as a float.
    """

    url = f"https://open.er-api.com/v6/latest/{base_currency.upper()}"

    response = requests.get(url)
    response.raise_for_status()  # Raises an exception if the request fails

    data = response.json()

    if data["result"] != "success":
        raise ValueError("Failed to fetch exchange rate.")

    return float(data["rates"][target_currency.upper()])





In [12]:
get_conversion_factor.invoke(
    {
        "base_currency": "USD",
        "target_currency": "INR"
    }
)

96.241592

In [13]:
@tool
def currency_converter(
    base_currency: str,
    target_currency: str,
    amount: float
) -> str:
    """
    Converts an amount from one currency to another.
    """

    conversion_factor = get_conversion_factor.invoke(
        {
            "base_currency": base_currency,
            "target_currency": target_currency
        }
    )

    converted_amount = amount * conversion_factor

    return (
        f"{amount} {base_currency.upper()} = "
        f"{converted_amount:.2f} {target_currency.upper()}"
    )

In [14]:
currency_converter.invoke(
    {
        "base_currency": "USD",
        "target_currency": "INR",
        "amount": 100
    }
)

'100.0 USD = 9624.16 INR'

In [15]:
#tool binding 

llm_with_tools=llm.bind_tools([get_conversion_factor,currency_converter])


In [ ]:
#tool calling

